# **The Implementation of Zero123++**




In [ ]:
!pip install --upgrade huggingface_hub==0.18.1   #old version that still worked
!pip install --upgrade git+https://github.com/huggingface/diffusers.git
!pip install --upgrade accelerate
!pip install pillow

ERROR: Ignored the following yanked versions: 0.8.0, 0.9.0.dev0, 0.9.0rc0, 0.16.1, 0.26.4
ERROR: Could not find a version that satisfies the requirement huggingface_hub==0.18.1 (from versions: 0.0.1, 0.0.2, 0.0.3rc1, 0.0.3rc2, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.1.0, 0.1.1, 0.1.2, 0.2.0, 0.2.1, 0.4.0, 0.5.0, 0.5.1, 0.6.0rc0, 0.6.0, 0.7.0rc0, 0.7.0, 0.8.0rc0, 0.8.0rc1, 0.8.0rc2, 0.8.0rc3, 0.8.0rc4, 0.8.1, 0.9.0rc2, 0.9.0rc3, 0.9.0, 0.9.1, 0.10.0rc0, 0.10.0rc1, 0.10.0rc3, 0.10.0, 0.10.1, 0.11.0rc0, 0.11.0rc1, 0.11.0, 0.11.1, 0.12.0rc0, 0.12.0, 0.12.1, 0.13.0rc0, 0.13.0rc1, 0.13.0, 0.13.1, 0.13.2, 0.13.3, 0.13.4, 0.14.0rc0, 0.14.0rc1, 0.14.0, 0.14.1, 0.15.0rc0, 0.15.0, 0.15.1, 0.16.0rc0, 0.16.2, 0.16.3, 0.16.4, 0.17.0rc0, 0.17.0, 0.17.1, 0.17.2, 0.17.3, 0.18.0rc0, 0.18.0, 0.19.0rc0, 0.19.0, 0.19.1, 0.19.2, 0.19.3, 0.19.4, 0.20.0rc0, 0.20.0rc1, 0.20.0, 0.20.1, 0.20.2, 0.20.3, 0.21.0rc0, 0.21.0, 0.21.1, 0.21.2,

In [ ]:
from PIL import Image

https://youtu.be/rK02eXm3mfI?si=cNqWo-WtwxuYxeEN

In [ ]:
# import os
# import torch
# from diffusers import DiffusionPipeline,  EulerAncestralDiscreteScheduler, ControlNetModel


# pipeline = DiffusionPipeline.from_pretrained("sudo-ai/zero123plus-v1.2",
#                                              custom_pipeline = "sudo-ai/zero123plus-pipeline",
#                                              torch_dtype = torch.float16)



# controlnet = ControlNetModel.from_pretrained("sudo-ai/controlnet-zp12-normal-gen-v1",torch_dtype = torch.float16)

# pipeline.add_controlnet(controlnet, conditioning_scale = 0.75)
# pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config, timestep_spacing = "trailing")
# pipeline.to("cuda")
# # pipeline.to("cpu")

# input_image = Image.open("/content/drive/MyDrive/zero123_extended/image_0001.jpg")


# os.makedirs("/content/drive/MyDrive/zero123_extended/colmap_project/images", exist_ok=True)

# # زوایای دید برای SFM
# angles = [i for i in range(0, 360, 30)]

# for angle in angles:
#     result = pipeline(input_image, view_angle=angle, num_inference_step = 75).images[0]
#     save_path = os.path.join("/content/drive/MyDrive/zero123_extended/colmap_project/images", f"image_{angle}.jpg")
#     result.save(save_path)
#     print(f"Saved image {save_path}")



# **For Croping Images to distinct identity and using COLMAP and Structure-from-Motion (SfM) and Multi-View Stereo (MVS) pipeline**

In [ ]:
import os
import torch
import requests
from PIL import Image
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

# لطفا این آدرس ها رو متناسب با سیستمت تغییر بذه
os.makedirs("/content/drive/MyDrive/zero123_extended/colmap_project", exist_ok=True)
OUTPUT_DIR = "/content/drive/MyDrive/zero123_extended/colmap_project"
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)


pipeline = DiffusionPipeline.from_pretrained("sudo-ai/zero123plus-v1.2",
                                             custom_pipeline = "sudo-ai/zero123plus-pipeline",
                                             torch_dtype = torch.float16)

pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config, timestep_spacing = "trailing")

pipeline.to("cuda:0")
# pipeline.to("cpu")


# لطفا آدرس تصویر را اینجا وارد کنید
img_url = "/content/drive/MyDrive/zero123_extended/image_0001.jpg"

cond = Image.open(img_url)


# شایان جان برای دقت بالاتر تا 90 میتونی inference رو تنطیم کنی
tiled_img = pipeline(cond, num_inference_step = 75).images[0]
tiled_img_path = os.path.join(IMAGES_DIR, "tiled_output.png")
tiled_img.save(tiled_img_path)
print(f"Saved tiled image to {tiled_img_path}")

cols, rows = 2, 3   # change this in other situation
tile_width, tile_height = tiled_img.size[0]//cols, tiled_img.size[1]//rows


for idx, (row, col) in enumerate([r , c] for r in range(rows) for c in range(cols)):
    left, upper = col * tile_width, row * tile_height
    right, lower = left + tile_width, upper + tile_height
    tile = tiled_img.crop((left, upper, right, lower))


    tile_path = os.path.join(IMAGES_DIR, f"view_{idx:02d}.png")
    tile.save(tile_path)
    print(f"Saved tile {idx} to {tile_path}")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/2.63k [00:00<?, ?B/s]

pipeline.py:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

tokenizer%2Fmerges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

vae%2Fconfig.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/8 [00:00<?, ?it/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

# **Structure From Motion With OpenMVG**

In [1]:
!git clone --recursive https://github.com/openMVG/openMVG.git

Cloning into 'openMVG'...
remote: Enumerating objects: 35162, done.
remote: Counting objects: 100% (3052/3052), done.
remote: Compressing objects: 100% (649/649), done.
remote: Total 35162 (delta 2772), reused 2437 (delta 2397), pack-reused 32110 (from 2)
Receiving objects: 100% (35162/35162), 30.92 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (25330/25330), done.
Submodule 'src/dependencies/cereal' (https://github.com/openMVG-thirdparty/cereal.git) registered for path 'src/dependencies/cereal'
Submodule 'src/dependencies/glfw' (https://github.com/elmindreda/glfw.git) registered for path 'src/dependencies/glfw'
Submodule 'src/dependencies/osi_clp' (https://github.com/openMVG-thirdparty/osi_clp.git) registered for path 'src/dependencies/osi_clp'
Cloning into '/content/openMVG/src/dependencies/cereal'...
remote: Enumerating objects: 6563, done.        
remote: Total 6563 (delta 0), reused 0 (delta 0), pack-reused 6563 (from 1)        
Receiving objects: 100% (6563/6563), 2.91 MiB | 6.

In [2]:
!mkdir -p openMVG_build && cd openMVG_build && cmake -DCMAKE_BUILD_TYPE=Release ../openMVG/src && make -j$(nproc)

CMake Deprecation Warning at CMakeLists.txt:8 (CMAKE_MINIMUM_REQUIRED):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- target changed from "" to "auto"
-- Detected CPU: zen
-- Performing Test check_c_compiler_flag__march_znver1
-- Performing

In [5]:
!cp -r /content/drive/MyDrive/openMVG_build  /content/


In [6]:
!cp -r /content/drive/MyDrive/zero123+sfm  /content/

پیدا کردن پارامترهای دوربین بر اساس مدل zero 123 +

In [7]:
from PIL import Image
import math

img_path = "/content/zero123+sfm/images/view_00.png"

im = Image.open(img_path)

width, height = im.size
print(f"the width of the image is {width} and the height is {height}")

fov_deg = 30
fov_rad = math.radians(fov_deg)
focal_length = (width/2) / ( math.tan(fov_rad / 2))
print(f"the focal length is {focal_length}")


cx = width/2
cy = height/2

print(f"the cx is {cx} and the cy is {cy}")


the width of the image is 320 and the height is 320
the focal length is 597.1281292110203
the cx is 160.0 and the cy is 160.0


In [18]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
-i /content/zero123+sfm \
-o /content/zero123+sfm/openMVG/sfm_data \
-k "597,0,160;0,597,160;0,0,1;"

INFO: [main_SfMInit_ImageListing.cpp:194]  You called : ./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing
--imageDirectory /content/zero123+sfm
--sensorWidthDatabase 
--outputDirectory /content/zero123+sfm/openMVG/sfm_data
--focal -1
--intrinsics 597,0,160;0,597,160;0,0,1;
--camera_model 3
--group_camera_model 1
--use_pose_prior 0
--prior_weights 1.0;1.0;1.0
--gps_to_xyz_method 0
ERROR: [main_SfMInit_ImageListing.cpp:45] 
 Missing ';' character
ERROR: [main_SfMInit_ImageListing.cpp:235] Invalid K matrix input


In [19]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
    -i /content/zero123+sfm \
    --sensorWidthDatabase openMVG/src/software/SfM/CameraSensorWidthDatabase.txt \
    --outputDirectory /content/zero123+sfm/openMVG/sfm_data \
    --focal -1 \
    --intrinsics "597,0,160;0,597,160;0,0,1" \
    --camera_model 3 \
    --group_camera_model 1 \
    --use_pose_prior 0 \
    --prior_weights "1.0;1.0;1.0" \
    --gps_to_xyz_method 0

INFO: [main_SfMInit_ImageListing.cpp:194]  You called : ./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing
--imageDirectory /content/zero123+sfm
--sensorWidthDatabase openMVG/src/software/SfM/CameraSensorWidthDatabase.txt
--outputDirectory /content/zero123+sfm/openMVG/sfm_data
--focal -1
--intrinsics 597,0,160;0,597,160;0,0,1
--camera_model 3
--group_camera_model 1
--use_pose_prior 1
--prior_weights 1.0;1.0;1.0
--gps_to_xyz_method 0
ERROR: [main_SfMInit_ImageListing.cpp:45] 
 Missing ';' character
ERROR: [main_SfMInit_ImageListing.cpp:235] Invalid K matrix input


In [20]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
    -i /content/zero123+sfm \
    -o /content/zero123+sfm/openMVG/sfm_data \
    -d openMVG/src/software/SfM/CameraSensorWidthDatabase.txt \
    -k "597,0,160;0,597,160;0,0,1" \
    --camera_model 3 \
    --group_camera_model 1 \
    --use_pose_prior 1 \
    --prior_weights "1.0;1.0;1.0" \
    --gps_to_xyz_method 0

INFO: [main_SfMInit_ImageListing.cpp:194]  You called : ./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing
--imageDirectory /content/zero123+sfm
--sensorWidthDatabase openMVG/src/software/SfM/CameraSensorWidthDatabase.txt
--outputDirectory /content/zero123+sfm/openMVG/sfm_data
--focal -1
--intrinsics 597,0,160;0,597,160;0,0,1
--camera_model 3
--group_camera_model 1
--use_pose_prior 1
--prior_weights 1.0;1.0;1.0
--gps_to_xyz_method 0
ERROR: [main_SfMInit_ImageListing.cpp:45] 
 Missing ';' character
ERROR: [main_SfMInit_ImageListing.cpp:235] Invalid K matrix input


In [22]:
!/openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
    -i /content/zero123+sfm/ \
    -o /content/zero123+sfm/openMVG/sfm_data \
    -d openMVG/src/softwares/SfM/CameraSensorWidthDatabase.txt

/bin/bash: line 1: /openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing: No such file or directory


In [23]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
    -i /content/zero123+sfm \
    -o /content/zero123+sfm/openMVG/sfm_data \
    -d openMVG/src/software/SfM/CameraSensorWidthDatabase.txt

INFO: [main_SfMInit_ImageListing.cpp:194]  You called : ./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing
--imageDirectory /content/zero123+sfm
--sensorWidthDatabase openMVG/src/software/SfM/CameraSensorWidthDatabase.txt
--outputDirectory /content/zero123+sfm/openMVG/sfm_data
--focal -1
--intrinsics 
--camera_model 3
--group_camera_model 1
--use_pose_prior 0
--prior_weights 1.0;1.0;1.0
--gps_to_xyz_method 0
ERROR: [ParseDatabase.hpp:52] Cannot read the database file: openMVG/src/software/SfM/CameraSensorWidthDatabase.txt
ERROR: [main_SfMInit_ImageListing.cpp:250] Invalid input database: openMVG/src/software/SfM/CameraSensorWidthDatabase.txt, please specify a valid file.


# **Worked**

In [52]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing \
    -i /content/zero123+sfm \
    -o /content/zero123+sfm/openMVG/sfm_data \
    -f 597.1281292110203

INFO: [main_SfMInit_ImageListing.cpp:194]  You called : ./openMVG_build/Linux-x86_64-Release/openMVG_main_SfMInit_ImageListing
--imageDirectory /content/zero123+sfm
--sensorWidthDatabase 
--outputDirectory /content/zero123+sfm/openMVG/sfm_data
--focal 597.128
--intrinsics 
--camera_model 3
--group_camera_model 1
--use_pose_prior 0
--prior_weights 1.0;1.0;1.0
--gps_to_xyz_method 0
Not a JPEG file: starts with 0x89 0x50
INFO: [loggerprogress.hpp:79] [- Listing images -] 100%
INFO: [main_SfMInit_ImageListing.cpp:475] SfMInit_ImageListing report:
listed #File(s): 7
usable #File(s) listed in sfm_data: 6
usable #Intrinsic(s) listed in sfm_data: 1


In [37]:
# !cp -r /content/zero123+sfm /content/drive/MyDrive

# **محاسبه ویژگی ها**

In [55]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeFeatures \
-i /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
-o /content/zero123+sfm/openMVG/ \
-m SIFT

INFO: [main_ComputeFeatures.cpp:120]  You called : 
./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeFeatures
--input_file /content/zero123+sfm/openMVG/sfm_data/sfm_data.json
--outdir /content/zero123+sfm/openMVG/
--describerMethod SIFT
--upright 0
--describerPreset NORMAL
--force 0
--numThreads 0

INFO: [loggerprogress.hpp:79] [- EXTRACT FEATURES -] 50%
INFO: [loggerprogress.hpp:79] [- EXTRACT FEATURES -] 100%
INFO: [main_ComputeFeatures.cpp:343] Task done in (s): 0


# **هماهنگ کردن تصاویر با هم MATCHING**

In [56]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeMatches \
-i /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
-o /content/zero123+sfm/openMVG/sfm_data


INFO: [main_ComputeMatches.cpp:112]  You called : 
./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeMatches
--input_file /content/zero123+sfm/openMVG/sfm_data/sfm_data.json
--output_file /content/zero123+sfm/openMVG/sfm_data
--pair_list 
Optional parameters:
--force 0
--ratio 0.8
--nearest_matching_method AUTO
--cache_size unlimited
--preemptive_feature_used/count 0 / 200
INFO: [loggerprogress.hpp:79] [- Regions Loading -] 50%
INFO: [loggerprogress.hpp:79] [- Regions Loading -] 100%
INFO: [main_ComputeMatches.cpp:215]  - PUTATIVE MATCHES - 
INFO: [main_ComputeMatches.cpp:236] Using FAST_CASCADE_HASHING_L2 matcher
INFO: [main_ComputeMatches.cpp:299] No input pair file set. Use exhaustive match by default.
INFO: [main_ComputeMatches.cpp:309] Running matching on #pairs: 15
INFO: [Cascade_Hashing_Matcher_Regions.cpp:238] Using the OPENMP thread interface
INFO: [loggerprogress.hpp:79] [- Matching -] 20%
INFO: [loggerprogress.hpp:79] [- Matching -] 40%
INFO: [loggerprogress.hpp:79] [

In [43]:
# !./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeFeatures \
#   --input_file /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
#   --outdir /content/zero123+sfm/openMVG/sfm_data \
#   --describerMethod SIFT \
#   --upright 0 \
#   --describerPreset NORMAL \
#   --force 1 \
#   --numThreads 0


INFO: [main_ComputeFeatures.cpp:120]  You called : 
./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeFeatures
--input_file /content/zero123+sfm/openMVG/sfm_data/sfm_data.json
--outdir /content/zero123+sfm/openMVG/sfm_data
--describerMethod SIFT
--upright 0
--describerPreset NORMAL
--force 1
--numThreads 0

INFO: [loggerprogress.hpp:79] [- EXTRACT FEATURES -] 50%
INFO: [loggerprogress.hpp:79] [- EXTRACT FEATURES -] 100%
INFO: [main_ComputeFeatures.cpp:343] Task done in (s): 0


In [48]:
# !./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeMatches \
#   -i /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
#   -o /content/zero123+sfm/openMVG/sfm_data


INFO: [main_ComputeMatches.cpp:112]  You called : 
./openMVG_build/Linux-x86_64-Release/openMVG_main_ComputeMatches
--input_file /content/zero123+sfm/openMVG/sfm_data/sfm_data.json
--output_file /content/zero123+sfm/openMVG/sfm_data
--pair_list 
Optional parameters:
--force 0
--ratio 0.8
--nearest_matching_method AUTO
--cache_size unlimited
--preemptive_feature_used/count 0 / 200
ERROR: [sfm_regions_provider.hpp:124] Invalid regions files for the view: /content/zero123+sfm/openMVG/sfm_data/view_00.png
ERROR: [sfm_regions_provider.hpp:124] Invalid regions files for the view: /content/zero123+sfm/openMVG/sfm_data/view_01.png
ERROR: [main_ComputeMatches.cpp:193] Cannot load view regions from: /content/zero123+sfm/openMVG.


In [67]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_GeometricFilter \
-i /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
-m /content/zero123+sfm/openMVG/sfm_data \
-o /content/zero123+sfm/openMVG/sfm_data/matches.f.bin


INFO: [main_GeometricFilter.cpp:131]  You called : 
./openMVG_build/Linux-x86_64-Release/openMVG_main_GeometricFilter
--input_file:        /content/zero123+sfm/openMVG/sfm_data/sfm_data.json
--matches:           /content/zero123+sfm/openMVG/sfm_data
--output_file:       /content/zero123+sfm/openMVG/sfm_data/matches.f.bin
Optional parameters: 
--input_pairs        
--output_pairs       
--force              false
--geometric_model    f
--guided_matching    0
--cache_size         unlimited
INFO: [loggerprogress.hpp:79] [- Regions Loading -] 50%
INFO: [loggerprogress.hpp:79] [- Regions Loading -] 100%
ERROR: [indMatch_utils.cpp:74] Unknown PairWiseMatches file extension: ().
INFO: [graph_stats.hpp:53] Graph statistics:
	#nodes: 6
	#cc: 6
	#singleton: 6
	Node degree statistics:	min: -1141170736, max: -1141170336, mean: 8101, median: -1141172223
INFO: [main_GeometricFilter.cpp:383] Task done in (s): 0
INFO: [main_GeometricFilter.cpp:386] 
 Export Adjacency Matrix of the pairwise's geometric

# **اجرای Structure From Motion (SFM)**

In [65]:
!./openMVG_build/Linux-x86_64-Release/openMVG_main_SfM \
-i /content/zero123+sfm/openMVG/sfm_data/sfm_data.json \
-m /content/zero123+sfm/openMVG/sfm_data \
-o /content/zero123+sfm/openMVG/out

INFO: [main_SfM.cpp:157] 
-----------------------------------------------------------
 Structure from Motion:
-----------------------------------------------------------
INFO: [loggerprogress.hpp:79] [- Features Loading -] 50%
INFO: [loggerprogress.hpp:79] [- Features Loading -] 100%
ERROR: [main_SfM.cpp:466] Cannot load the match file.


In [68]:
!cp -r /content/zero123+sfm_ /content/drive/MyDrive